# 01 — Data Loading, Merging & Cleaning

## Context
This notebook handles the foundational data layer for the A/B test analysis. Before any statistical work can be trusted, the data pipeline must be clean, well-understood, and explicitly validated.

We have three data sources:
- **Demographics** (`df_final_demo.csv`): client-level attributes — age, balance, tenure, gender, account counts
- **Web funnel behavior** (`df_final_web_data_pt_1.csv` + `pt_2.csv`): event-level digital footprint across a 5-step funnel
- **Experiment assignment** (`df_final_experiment_clients.csv`): which clients are in Test vs. Control

### Key design decision: unit of analysis
The experiment was **randomized at `client_id` level**, so all analysis must aggregate to that level. Using `visitor_id` or `visit_id` as the analysis unit would treat correlated observations from the same client as independent — inflating the effective sample size and producing artificially narrow confidence intervals. This notebook enforces that discipline from the start.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


## 1. Load Raw Data

In [10]:
# Load data from relative path
df_demo = pd.read_csv('../data/df_final_demo.csv')
df_web = pd.concat([pd.read_csv('../data/df_final_web_data_pt_1.csv'), pd.read_csv('../data/df_final_web_data_pt_2.csv')], ignore_index=True)
df_exp = pd.read_csv('../data/df_final_experiment_clients.csv')

print(f'Demographics:       {df_demo.shape[0]:,} rows | {df_demo.shape[1]} columns')
print(f'Web funnel (pt1):   {df_web_pt1.shape[0]:,} rows')

FileNotFoundError: [Errno 2] No such file or directory: '../data/df_final_demo.csv'

In [11]:
# --- Load demographics ---
df_demo = pd.read_csv('../data/df_final_demo.csv')

# --- Load and combine web funnel data (split across two files) ---
df_web_pt1 = pd.read_csv('../data/df_final_web_data_pt_1.csv')
df_web_pt2 = pd.read_csv('../data/df_final_web_data_pt_2.csv')
df_web = pd.concat([df_web_pt1, df_web_pt2], ignore_index=True)

# --- Load experiment assignment ---
df_exp = pd.read_csv('../data/df_final_experiment_clients.csv')

print(f'Demographics:       {df_demo.shape[0]:,} rows | {df_demo.shape[1]} columns')
print(f'Web funnel (pt1):   {df_web_pt1.shape[0]:,} rows')
print(f'Web funnel (pt2):   {df_web_pt2.shape[0]:,} rows')
print(f'Web funnel (total): {df_web.shape[0]:,} rows | {df_web.shape[1]} columns')
print(f'Experiment clients: {df_exp.shape[0]:,} rows | {df_exp.shape[1]} columns')

Demographics:       70,609 rows | 9 columns
Web funnel (pt1):   343,141 rows
Web funnel (pt2):   412,264 rows
Web funnel (total): 755,405 rows | 5 columns
Experiment clients: 70,609 rows | 2 columns


## 2. Initial Inspection

In [3]:
print('=== DEMOGRAPHICS ===')
display(df_demo.head(3))
display(df_demo.dtypes)
print('\nMissing values:')
print(df_demo.isnull().sum())

=== DEMOGRAPHICS ===


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,836976,6.00,73.00,60.50,U,2.00,45105.30,6.00,9.00
1,2304905,7.00,94.00,58.00,U,2.00,110860.30,6.00,9.00
2,1439522,5.00,64.00,32.00,U,2.00,52467.79,6.00,9.00


client_id             int64
clnt_tenure_yr      float64
clnt_tenure_mnth    float64
clnt_age            float64
gendr                object
num_accts           float64
bal                 float64
calls_6_mnth        float64
logons_6_mnth       float64
dtype: object


Missing values:
client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64


In [4]:
print('=== WEB FUNNEL ===')
display(df_web.head(5))
print('\nUnique process steps:')
print(df_web['process_step'].value_counts())
print('\nMissing values:')
print(df_web.isnull().sum())

=== WEB FUNNEL ===


,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04



Unique process steps:
process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64

Missing values:
client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64


In [5]:
print('=== EXPERIMENT ASSIGNMENT ===')
display(df_exp.head(3))
print('\nVariation distribution:')
print(df_exp['Variation'].value_counts())

=== EXPERIMENT ASSIGNMENT ===


,client_id,Variation
0,9988021,Test
1,8320017,Test
2,4033851,Control



Variation distribution:
Variation
Test       26968
Control    23532
Name: count, dtype: int64


## 3. Cleaning — Demographics

Handle missing values and standardize gender coding.

In [6]:
df_demo_clean = df_demo.copy()

# Standardize gender: map 'U' (unknown) → NaN for analysis purposes
df_demo_clean['gendr'] = df_demo_clean['gendr'].replace('U', np.nan)

# Drop rows with missing client_id — cannot be joined without it
missing_client = df_demo_clean['client_id'].isnull().sum()
print(f'Rows with missing client_id: {missing_client}')
df_demo_clean = df_demo_clean.dropna(subset=['client_id'])

# Check for duplicate client_ids in demographics
dupe_clients = df_demo_clean['client_id'].duplicated().sum()
print(f'Duplicate client_ids in demographics: {dupe_clients}')

print(f'\nClean demographics shape: {df_demo_clean.shape}')

Rows with missing client_id: 0
Duplicate client_ids in demographics: 0

Clean demographics shape: (70609, 9)


## 4. Cleaning — Web Funnel

Parse datetime, enforce step ordering, and handle duplicates. A key edge case: the same `visit_id` can have repeated entries for the same step (e.g., page refreshes). We keep the earliest timestamp per step per visit.

In [7]:
df_web_clean = df_web.copy()

# Parse datetime
df_web_clean['date_time'] = pd.to_datetime(df_web_clean['date_time'])

# Define step order for sorting and analysis
STEP_ORDER = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
step_rank = {step: i for i, step in enumerate(STEP_ORDER)}
df_web_clean['step_rank'] = df_web_clean['process_step'].map(step_rank)

# Drop rows with unrecognized steps
unknown_steps = df_web_clean['step_rank'].isnull().sum()
print(f'Rows with unrecognized process_step: {unknown_steps}')
df_web_clean = df_web_clean.dropna(subset=['step_rank'])

# Deduplicate: keep earliest timestamp per (visit_id, process_step)
df_web_clean = (
    df_web_clean
    .sort_values('date_time')
    .drop_duplicates(subset=['visit_id', 'process_step'], keep='first')
)

print(f'Web funnel rows after deduplication: {df_web_clean.shape[0]:,}')

Rows with unrecognized process_step: 0
Web funnel rows after deduplication: 553,417


## 5. Experiment Date Range

Derive the actual test window from the data — do not hardcode dates.

In [8]:
date_min = df_web_clean['date_time'].min()
date_max = df_web_clean['date_time'].max()
duration_days = (date_max - date_min).days

print(f'Experiment start:  {date_min.date()}')
print(f'Experiment end:    {date_max.date()}')
print(f'Duration:          {duration_days} days (~{duration_days // 30} months)')

Experiment start:  2017-03-15
Experiment end:    2017-06-20
Duration:          97 days (~3 months)


## 6. Build Client-Level Analysis Table

Aggregate funnel behavior to `client_id` level — the unit of randomization and analysis.

In [9]:
# Per client: did they reach 'confirm'? What was their max step?
client_funnel = (
    df_web_clean
    .groupby('client_id')
    .agg(
        max_step_rank=('step_rank', 'max'),
        num_visits=('visit_id', 'nunique'),
        num_visitors=('visitor_id', 'nunique'),
        first_seen=('date_time', 'min'),
        last_seen=('date_time', 'max')
    )
    .reset_index()
)

# Map back to step name
rank_step = {v: k for k, v in step_rank.items()}
client_funnel['max_step'] = client_funnel['max_step_rank'].map(rank_step)

# Binary completion flag: reached 'confirm' (rank 4)
CONFIRM_RANK = step_rank['confirm']
client_funnel['completed'] = (client_funnel['max_step_rank'] == CONFIRM_RANK).astype(int)

print(f'Clients with funnel data: {client_funnel.shape[0]:,}')
print(f'Overall completion rate: {client_funnel["completed"].mean():.1%}')
display(client_funnel.head())

Clients with funnel data: 119,425
Overall completion rate: 67.3%


,client_id,max_step_rank,num_visits,num_visitors,first_seen,last_seen,max_step,completed
0,169,4,1,1,2017-04-12 20:19:36,2017-04-12 20:23:09,confirm,1
1,336,0,1,1,2017-06-01 07:26:55,2017-06-01 07:26:55,start,0
2,546,4,1,1,2017-06-17 10:03:29,2017-06-17 10:05:42,confirm,1
3,555,4,1,1,2017-04-15 12:57:56,2017-04-15 13:00:34,confirm,1
4,647,4,1,1,2017-04-12 15:41:28,2017-04-12 15:47:45,confirm,1


## 7. Merge All Sources

In [10]:
# Merge experiment assignment → funnel → demographics
# Left join on experiment clients — only include clients in the experiment
df_analysis = (
    df_exp
    .merge(client_funnel, on='client_id', how='left')
    .merge(df_demo_clean, on='client_id', how='left')
)

print(f'Analysis table shape: {df_analysis.shape}')
print(f'\nVariation split:')
print(df_analysis['Variation'].value_counts())
print(f'\nClients with no funnel data: {df_analysis["max_step_rank"].isnull().sum():,}')

Analysis table shape: (70609, 17)

Variation split:
Variation
Test       26968
Control    23532
Name: count, dtype: int64

Clients with no funnel data: 441


In [11]:
# Clients assigned to the experiment but with no web events
# These are clients who never visited — treat completed = 0
df_analysis['completed'] = df_analysis['completed'].fillna(0).astype(int)
df_analysis['max_step_rank'] = df_analysis['max_step_rank'].fillna(-1).astype(int)
df_analysis['max_step'] = df_analysis['max_step'].fillna('no_visit')

print('Final analysis table — null check:')
print(df_analysis.isnull().sum())

Final analysis table — null check:
client_id               0
Variation           20109
max_step_rank           0
num_visits            441
num_visitors          441
first_seen            441
last_seen             441
max_step                0
completed               0
clnt_tenure_yr         14
clnt_tenure_mnth       14
clnt_age               15
gendr               24136
num_accts              14
bal                    14
calls_6_mnth           14
logons_6_mnth          14
dtype: int64


## 8. Save Clean Data

In [12]:
import os
os.makedirs('output', exist_ok=True)

df_analysis.to_csv('../output/df_analysis_client_level.csv', index=False)
df_web_clean.to_csv('../output/df_web_clean.csv', index=False)

print('Saved:')
print('  output/df_analysis_client_level.csv  — main analysis table (one row per client)')
print('  output/df_web_clean.csv              — cleaned event-level web data')

Saved:
  output/df_analysis_client_level.csv  — main analysis table (one row per client)
  output/df_web_clean.csv              — cleaned event-level web data


## Summary

| Item | Value |
|---|---|
| Experiment clients (total) | 70,609 |
| Assigned to Test or Control | 50,500 (26,968 Test / 23,532 Control) |
| Unassigned clients | 20,109 — excluded from analysis (no group label) |
| Date range | 97 days (~3 months) |
| Analysis unit | `client_id` |
| Analysis table shape | (70,609 × 17) |
| Completion definition | Reached `confirm` step at least once |
| Clients with no funnel data | 441 — treated as `completed = 0` |